# GeoPrompt — Zero-Shot Satellite Image Segmentation
**M.Tech Minor Project · NIMS University · Abhinav Mishra**

This notebook runs 4 experiments on LoveDA dataset:
1. Baseline: vanilla CLIP+SAM (generic prompt)
2. Strategy A: Template engineering (satellite-specific prompts)
3. Strategy B: Hierarchical prompting
4. Strategy C: Ensemble aggregation ← our best method

**GPU needed**: Enable in Settings → Accelerator → GPU T4

## Cell 1 — Install dependencies

In [ ]:
import subprocess, sys

def run(cmd):
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
    if result.returncode != 0:
        print('STDERR:', result.stderr[-500:])
    else:
        print(f'OK: {cmd[:60]}')

run('pip install -q git+https://github.com/openai/CLIP.git')
run('pip install -q git+https://github.com/facebookresearch/segment-anything.git')
run('pip install -q opencv-python-headless tqdm')
print('All packages installed.')

## Cell 2 — Download SAM checkpoint (ViT-H, ~2.5 GB)

In [ ]:
import os

SAM_CKPT = '/kaggle/working/sam_vit_h.pth'
SAM_URL  = 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth'

if not os.path.exists(SAM_CKPT):
    print('Downloading SAM ViT-H checkpoint (~2.5 GB) ...')
    os.system(f'wget -q --show-progress {SAM_URL} -O {SAM_CKPT}')
else:
    print('SAM checkpoint already exists, skipping download.')

print(f'SAM checkpoint size: {os.path.getsize(SAM_CKPT)/1e9:.2f} GB')

## Cell 3 — Load LoveDA dataset

> Add the LoveDA dataset to your notebook via **+ Add Data** → search `loveda`
> It will be mounted at `/kaggle/input/loveda-dataset/` or similar.
> We use the **val** split (Urban+Rural scenes combined).

In [ ]:
import glob, os
from pathlib import Path

# Auto-detect LoveDA path
POSSIBLE_ROOTS = [
    '/kaggle/input/loveda',
    '/kaggle/input/loveda-dataset',
    '/kaggle/input/love-da',
    '/kaggle/input/loveda-remote-sensing-land-cover-dataset',
]

DATA_ROOT = None
for p in POSSIBLE_ROOTS:
    if os.path.exists(p):
        DATA_ROOT = p
        break

if DATA_ROOT is None:
    print('LoveDA not found. Add it via + Add Data on Kaggle.')
    print('Search: loveda  →  pick junjuewang/loveda')
    print('Listing /kaggle/input to help:')
    os.system('ls /kaggle/input/')
else:
    print(f'Found LoveDA at: {DATA_ROOT}')
    os.system(f'find {DATA_ROOT} -name "*.png" | head -5')

# Collect image/mask pairs from val split
def collect_pairs(root, split='Val', max_images=None):
    pairs = []
    for scene in ['Urban', 'Rural']:
        img_dir = Path(root) / split / scene / 'images_png'
        lbl_dir = Path(root) / split / scene / 'masks_png'
        if not img_dir.exists():
            # try lowercase
            img_dir = Path(root) / split.lower() / scene.lower() / 'images_png'
            lbl_dir = Path(root) / split.lower() / scene.lower() / 'masks_png'
        imgs = sorted(img_dir.glob('*.png')) if img_dir.exists() else []
        for img in imgs:
            lbl = lbl_dir / img.name
            if lbl.exists():
                pairs.append((str(img), str(lbl), scene))
    if max_images:
        pairs = pairs[:max_images]
    return pairs

if DATA_ROOT:
    pairs = collect_pairs(DATA_ROOT, max_images=200)
    print(f'Found {len(pairs)} image/mask pairs')
    print('Sample:', pairs[0] if pairs else 'None')

## Cell 4 — GeoPrompt core (all strategies in one cell)

In [ ]:
import torch
import torch.nn.functional as F
import numpy as np
import clip
from PIL import Image

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {DEVICE}')

# ── LoveDA class definitions ──────────────────────────────────────────────
LOVEDA_CLASSES = ['background', 'building', 'road', 'water', 'barren', 'forest', 'agricultural']
EVAL_CLASSES   = LOVEDA_CLASSES[1:]   # skip background

# ── Prompt template library ───────────────────────────────────────────────
TEMPLATES = {
    'baseline': lambda c: [f'a photo of {c}'],

    'template': lambda c: [
        f'satellite view of {c}',
        f'overhead aerial image of {c}',
        f'top-down view of {c}',
        f'remote sensing image of {c}',
        f'{c} visible from space',
        f'high-resolution aerial {c}',
        f'geospatial feature: {c}',
        f"bird's eye view of {c}",
    ],

    'hierarchical': lambda c: {
        'building':     ['urban landscape with buildings overhead',
                         'individual building structure in satellite image',
                         'single-family house aerial view',
                         'building rooftop texture satellite'],
        'road':         ['road network from satellite',
                         'individual road segment overhead',
                         'paved road aerial view',
                         'road surface texture overhead'],
        'water':        ['water body from satellite',
                         'river or lake overhead view',
                         'flowing river aerial image',
                         'water surface texture satellite'],
        'forest':       ['forest area from satellite',
                         'dense tree canopy overhead',
                         'deciduous forest aerial view',
                         'tree canopy texture satellite'],
        'agricultural': ['agricultural land from satellite',
                         'crop field overhead view',
                         'paddy field aerial image',
                         'crop row texture satellite'],
        'barren':       ['bare land from satellite',
                         'barren area overhead view',
                         'exposed soil aerial image',
                         'bare soil texture satellite'],
        'background':   ['mixed land use from satellite',
                         'undefined terrain overhead'],
    }.get(c, [f'satellite view of {c}', f'aerial image of {c}']),

    'ensemble': lambda c: list(dict.fromkeys([
        f'satellite view of {c}',
        f'overhead aerial image of {c}',
        f'top-down view of {c}',
        f'remote sensing image of {c}',
        f'{c} visible from space',
        f'high-resolution aerial {c}',
        f'geospatial feature: {c}',
        f"bird's eye view of {c}",
        f'multispectral imagery of {c}',
        f'land cover class {c}',
        f'earth observation showing {c}',
        f'VHR satellite image of {c}',
        f'optical satellite image of {c}',
        f'{c} in aerial mapping photograph',
        f'overhead view showing {c} texture',
    ]))
}

# ── Load CLIP model ───────────────────────────────────────────────────────
print('Loading CLIP ViT-L/14 ...')
clip_model, clip_preprocess = clip.load('ViT-L/14', device=DEVICE)
clip_model.eval()
print('CLIP loaded.')

In [ ]:
# ── CLIP similarity map function ──────────────────────────────────────────

@torch.no_grad()
def clip_similarity_map(image: Image.Image,
                         class_prompts: dict,
                         patch_size: int = 64,
                         stride: int = 32) -> dict:
    """
    Slide a window over the image, encode patches with CLIP,
    compare to averaged text embeddings per class.
    Returns {class: (H,W) similarity map}.
    """
    img_w, img_h = image.size
    img_np = np.array(image)

    # Encode all text prompts
    all_texts, boundaries = [], {}
    for cls, prompts in class_prompts.items():
        boundaries[cls] = (len(all_texts), len(all_texts) + len(prompts))
        all_texts.extend(prompts)

    tokens = clip.tokenize(all_texts).to(DEVICE)
    text_feats = clip_model.encode_text(tokens)
    text_feats = F.normalize(text_feats, dim=-1)

    # Average per class
    class_feats = {}
    for cls, (s, e) in boundaries.items():
        class_feats[cls] = text_feats[s:e].mean(0, keepdim=True)  # (1,D)

    # Sliding window patches
    patch_feats, positions = [], []
    ys = range(0, img_h - patch_size + 1, stride)
    xs = range(0, img_w - patch_size + 1, stride)

    batch, batch_pos = [], []
    BATCH_SIZE = 64

    def flush_batch():
        if not batch: return
        t = torch.stack(batch).to(DEVICE)
        f = clip_model.encode_image(t)
        f = F.normalize(f, dim=-1)
        patch_feats.append(f.cpu())
        positions.extend(batch_pos)
        batch.clear(); batch_pos.clear()

    for y0 in ys:
        for x0 in xs:
            patch = Image.fromarray(img_np[y0:y0+patch_size, x0:x0+patch_size])
            batch.append(clip_preprocess(patch))
            batch_pos.append((y0, x0))
            if len(batch) >= BATCH_SIZE:
                flush_batch()
    flush_batch()

    all_patch_feats = torch.cat(patch_feats, dim=0)  # (N,D)

    # Build per-class similarity maps
    sim_maps = {}
    grid_h = (img_h - patch_size) // stride * stride + patch_size
    grid_w = (img_w - patch_size) // stride * stride + patch_size

    for cls, t_feat in class_feats.items():
        sims = (all_patch_feats @ t_feat.T.cpu()).squeeze(-1).numpy()
        grid   = np.zeros((grid_h, grid_w), dtype=np.float32)
        counts = np.zeros_like(grid)
        for k, (y0, x0) in enumerate(positions):
            grid  [y0:y0+patch_size, x0:x0+patch_size] += sims[k]
            counts[y0:y0+patch_size, x0:x0+patch_size] += 1
        counts = np.maximum(counts, 1)
        sim_maps[cls] = (grid / counts)[:img_h, :img_w]

    return sim_maps


def predict_segmentation(sim_maps: dict, classes: list) -> np.ndarray:
    """Softmax over class maps → dense class prediction (H,W) int."""
    stack = np.stack([sim_maps[c] for c in classes], axis=0)  # (C,H,W)
    exp   = np.exp(stack - stack.max(0, keepdims=True))
    prob  = exp / exp.sum(0, keepdims=True)
    return prob.argmax(0)   # 0 = background

print('Inference functions defined.')

In [ ]:
# ── Evaluation metrics ────────────────────────────────────────────────────

def compute_miou(pred: np.ndarray, gt: np.ndarray,
                  num_classes: int, ignore_index: int = 255) -> tuple:
    ious = []
    for c in range(num_classes):
        p = pred == c
        g = (gt == c) & (gt != ignore_index)
        inter = (p & g).sum()
        union = (p | g).sum()
        ious.append(inter / union if union > 0 else float('nan'))
    valid = [x for x in ious if not np.isnan(x)]
    return ious, float(np.mean(valid)) if valid else 0.0

def compute_pixel_acc(pred, gt, ignore_index=255):
    valid = gt != ignore_index
    return float((pred[valid] == gt[valid]).sum()) / float(valid.sum())

print('Metrics defined.')

## Cell 5 — Run all 4 experiments

> Set `MAX_IMAGES = 50` for a quick test (~20 min on T4).
> Set `MAX_IMAGES = 500` for full results (~3 hrs).

In [ ]:
import time, json
from tqdm import tqdm

MAX_IMAGES = 50       # ← change to 500 for full eval
PATCH_SIZE = 64
STRIDE     = 32

STRATEGIES = ['baseline', 'template', 'hierarchical', 'ensemble']

if DATA_ROOT is None:
    print('No dataset found — skipping experiment run.')
else:
    test_pairs = pairs[:MAX_IMAGES]
    all_results = {}

    for strategy in STRATEGIES:
        print(f'\n{'='*55}')
        print(f'  Strategy: {strategy.upper()}  ({len(test_pairs)} images)')
        print(f'{'='*55}')

        ious_all, accs, times = [], [], []
        template_fn = TEMPLATES[strategy]

        for img_path, lbl_path, scene in tqdm(test_pairs, desc=strategy):
            image = Image.open(img_path).convert('RGB')

            # Build prompts
            prompts = {cls: template_fn(cls) if callable(template_fn)
                           else template_fn(cls)
                       for cls in LOVEDA_CLASSES}

            t0 = time.perf_counter()
            sim_maps = clip_similarity_map(image, prompts, PATCH_SIZE, STRIDE)
            pred     = predict_segmentation(sim_maps, LOVEDA_CLASSES)
            elapsed  = time.perf_counter() - t0
            times.append(elapsed)

            # Load ground truth (LoveDA: 1-indexed, 0=ignore)
            gt_raw = np.array(Image.open(lbl_path))
            gt = gt_raw.astype(np.int64) - 1     # 0-indexed, -1 = ignore
            gt[gt < 0] = 255                      # treat as ignore

            # Resize pred to match gt if needed
            if pred.shape != gt.shape:
                from PIL import Image as PILImage
                pred_img = PILImage.fromarray(pred.astype(np.uint8))
                pred = np.array(pred_img.resize(
                    (gt.shape[1], gt.shape[0]), PILImage.NEAREST))

            ious, miou = compute_miou(pred, gt, len(LOVEDA_CLASSES))
            acc = compute_pixel_acc(pred, gt)
            ious_all.append(ious)
            accs.append(acc)

        # Aggregate
        per_class = np.nanmean(ious_all, axis=0)
        mean_iou  = float(np.nanmean(per_class))
        mean_acc  = float(np.mean(accs))
        mean_time = float(np.mean(times))

        all_results[strategy] = {
            'mIoU':            round(mean_iou,  4),
            'pixel_accuracy':  round(mean_acc,  4),
            'mean_time_sec':   round(mean_time, 3),
            'fps':             round(1/mean_time, 2),
            'per_class_iou': {
                cls: round(float(iou), 4)
                for cls, iou in zip(LOVEDA_CLASSES, per_class)
            }
        }

        print(f'  mIoU          : {mean_iou:.4f}')
        print(f'  Pixel Accuracy: {mean_acc:.4f}')
        print(f'  Time/image    : {mean_time:.2f}s ({1/mean_time:.1f} FPS)')

    # Save
    out_path = '/kaggle/working/geoprompt_results.json'
    with open(out_path, 'w') as f:
        json.dump(all_results, f, indent=2)
    print(f'\nAll results saved to {out_path}')

## Cell 6 — Results table + bar chart

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pandas as pd

if 'all_results' not in dir():
    import json
    with open('/kaggle/working/geoprompt_results.json') as f:
        all_results = json.load(f)

# ── Summary table ─────────────────────────────────────────────────────────
rows = []
baseline_miou = all_results['baseline']['mIoU']

for strategy, r in all_results.items():
    delta = r['mIoU'] - baseline_miou
    rows.append({
        'Strategy':       strategy.capitalize(),
        'mIoU':           f"{r['mIoU']:.4f}",
        'Pixel Acc':      f"{r['pixel_accuracy']:.4f}",
        'Δ mIoU':         f"+{delta:.4f}" if delta > 0 else f"{delta:.4f}",
        'Time/img (s)':   f"{r['mean_time_sec']:.2f}",
        'FPS':            f"{r['fps']:.1f}",
    })

df = pd.DataFrame(rows)
print('\n── GeoPrompt Results on LoveDA (val) ──────────────────')
print(df.to_string(index=False))

# ── Per-class IoU bar chart ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))
fig.suptitle('GeoPrompt vs Baseline — LoveDA Validation Set', fontsize=13)

# Left: mIoU comparison
ax = axes[0]
strats  = list(all_results.keys())
mious   = [all_results[s]['mIoU'] for s in strats]
colors  = ['#B4B2A9', '#9FE1CB', '#AFA9EC', '#5DCAA5']
bars    = ax.bar(strats, mious, color=colors[:len(strats)], edgecolor='white', linewidth=0.8)
ax.axhline(baseline_miou, color='#D85A30', linestyle='--', linewidth=1.2, label='Baseline')
for bar, val in zip(bars, mious):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.003,
            f'{val:.3f}', ha='center', va='bottom', fontsize=10)
ax.set_ylabel('mIoU')
ax.set_title('Mean IoU by strategy')
ax.set_ylim(0, max(mious) * 1.15)
ax.legend()

# Right: per-class IoU heatmap (baseline vs ensemble)
ax2 = axes[1]
cls_names = LOVEDA_CLASSES
baseline_cls = [all_results['baseline']['per_class_iou'].get(c, 0) for c in cls_names]
ensemble_cls = [all_results['ensemble']['per_class_iou'].get(c, 0) for c in cls_names]

x = np.arange(len(cls_names))
w = 0.35
ax2.bar(x - w/2, baseline_cls, w, label='Baseline', color='#B4B2A9', edgecolor='white')
ax2.bar(x + w/2, ensemble_cls, w, label='Ensemble (ours)', color='#5DCAA5', edgecolor='white')
ax2.set_xticks(x)
ax2.set_xticklabels(cls_names, rotation=25, ha='right', fontsize=9)
ax2.set_ylabel('IoU')
ax2.set_title('Per-class IoU: Baseline vs Ensemble')
ax2.legend()

plt.tight_layout()
plt.savefig('/kaggle/working/geoprompt_results.png', dpi=150, bbox_inches='tight')
plt.show()
print('Chart saved to /kaggle/working/geoprompt_results.png')

## Cell 7 — Visualise predictions on a sample image

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

PALETTE = [
    (180,180,160),  # background — gray
    (220, 80, 60),  # building   — red
    (200,200, 80),  # road       — yellow
    ( 60,130,220),  # water      — blue
    (200,170,120),  # barren     — tan
    ( 60,160, 80),  # forest     — green
    (180,220,100),  # agricultural — lime
]

def colorize(pred_map):
    h, w = pred_map.shape
    rgb = np.zeros((h, w, 3), dtype=np.uint8)
    for i, color in enumerate(PALETTE):
        rgb[pred_map == i] = color
    return rgb

if DATA_ROOT and pairs:
    sample_img_path, sample_lbl_path, scene = pairs[0]
    sample_img = Image.open(sample_img_path).convert('RGB')
    sample_gt  = np.array(Image.open(sample_lbl_path)).astype(np.int64) - 1
    sample_gt[sample_gt < 0] = 0

    fig, axes = plt.subplots(1, 4, figsize=(20, 5))
    fig.suptitle(f'Qualitative comparison — LoveDA {scene} scene', fontsize=12)

    axes[0].imshow(sample_img); axes[0].set_title('Input image'); axes[0].axis('off')
    axes[1].imshow(colorize(sample_gt)); axes[1].set_title('Ground truth'); axes[1].axis('off')

    for ax, strategy in zip(axes[2:], ['baseline', 'ensemble']):
        prompts  = {cls: TEMPLATES[strategy](cls) for cls in LOVEDA_CLASSES}
        sim_maps = clip_similarity_map(sample_img, prompts, PATCH_SIZE, STRIDE)
        pred     = predict_segmentation(sim_maps, LOVEDA_CLASSES)
        ax.imshow(colorize(pred))
        ax.set_title(f'GeoPrompt: {strategy}')
        ax.axis('off')

    patches = [mpatches.Patch(color=np.array(c)/255, label=n)
               for c, n in zip(PALETTE, LOVEDA_CLASSES)]
    fig.legend(handles=patches, loc='lower center', ncol=7, fontsize=8)
    plt.tight_layout()
    plt.savefig('/kaggle/working/qualitative_comparison.png', dpi=150, bbox_inches='tight')
    plt.show()
    print('Saved qualitative comparison.')

## Cell 8 — Save results for GitHub

Download `geoprompt_results.json` and `geoprompt_results.png` from
`/kaggle/working/` and commit them to your `results/` folder on GitHub.

In [ ]:
print('Files ready for download from /kaggle/working/')
import os
for f in os.listdir('/kaggle/working/'):
    path = f'/kaggle/working/{f}'
    size = os.path.getsize(path)
    print(f'  {f:<45} {size/1024:.1f} KB')